In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import numpy as np

R_CATHODE = 46.29e-3 / 2   # m  — cathode sphere radius
R_WALL    = 280e-3   / 2   # m  — chamber wall radius

radius = np.linspace(R_CATHODE, R_WALL, 400)  # 400 radial points from cathode to wall

class PlasmaMLP(nn.Module):
    '''
    input: radius, Power, Voltage, Current,Pressure , rat1, rat2, rat3, rat4, rat5  (10 features)
    output: ne, Te (2 features)
    '''
    def __init__(self, input_size=10, hidden_size=128, output_size=2):
        super(PlasmaMLP, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(input_size, hidden_size // 2),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_size // 2, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, output_size)
        )

    def forward(self, x):
        return self.network(x)
    

def train_plasma_model(df, epochs=100, batch_size=32):
    X = df[['radius', 'power', 'voltage', 'current', 'pressure', 
            'rat1', 'rat2', 'rat3', 'rat4', 'rat5']].values
    
    y_ne = np.log10(df['ne'].values).reshape(-1, 1)
    y_te = df['te'].values.reshape(-1, 1)
    y = np.hstack([y_ne, y_te])

    scaler_x = StandardScaler().fit(X)
    scaler_y = StandardScaler().fit(y)

    X_train = torch.tensor(scaler_x.transform(X), dtype=torch.float32)
    y_train = torch.tensor(scaler_y.transform(y), dtype=torch.float32)

    dataset = TensorDataset(X_train, y_train)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # --- Training Setup ---
    model = PlasmaMLP()
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.MSELoss()

    model.train()
    for epoch in range(epochs):
        for batch_x, batch_y in loader:
            optimizer.zero_grad()
            outputs = model(batch_x)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()
            
    return model, scaler_x, scaler_y
    


In [1]:
import json

with open('training_data.json', 'r', encoding='utf-8') as file:
    data = json.load(file)

print(data)

{'0': {'Current': 60.0, 'Voltage': 38.0, 'Pressure': 51.5, 'Peak_Ratios': [1.5045456822222023, 2.256411094945, 1.0636364205555506, 1.6467667964265882, 0.3960064404638234, nan], 'Electron_Temperatures': [1.583084675167139, 1.51607507656224, 1.4127856885505627, 1.354908571873652, 1.3424447896415268, 1.3390906419226885, 1.3357699517576425, 1.3324823854079857, 1.3292276124643476, 1.3260053058131844, 1.3228151416039036, 1.3196567992163155, 1.3165299612284098, 1.3134343133844544, 1.3103695445634118, 1.3073353467476694, 1.304331414992086, 1.3013574473933387, 1.298413145059587, 1.295498212080429, 1.2926123554971625, 1.2897552852733432, 1.286926714265633, 1.2841263581949445, 1.2813539356178665, 1.2786091678983809, 1.275891779179858, 1.273201496357332, 1.2705380490500544, 1.267901169574317, 1.2652905929165525, 1.2627060567066994, 1.26014730119183, 1.2576140692100477, 1.2551061061646402, 1.2526231599984925, 1.2501649811687545, 1.2477313226217586, 1.2453219397681954, 1.2429365904585272, 1.24057503